# 07 · Evaluation deep dive: uncertainty, fairness, warning time, explanations

**Who this is for:** you, after notebook 06 (which registered the `@champion` model).
**You should already know:** time splits, AUPRC, the alert budget and NEWS2 (notebook 06).

**By the end you will be able to:**
- put **confidence intervals** on every metric, with a bootstrap that respects patients
- tell whether the champion beats NEWS2 **for real** or only by luck
- compare models **at the same alert load**, and spot alert-load drift
- check performance **per slice** (site, age, sex, ward) with one shared threshold
- measure **how many hours of warning** an alert gives before an escalation
- explain any single prediction with **SHAP** contributions, grouped so a nurse can read them

**Outline**
1. Setup: load `@champion` from MLflow
2. Recap: the single numbers from notebook 06
3. How sure are we? The cluster bootstrap
4. Alert-load drift: a label-free early warning
5. Fairness: slices
6. Warning time
7. Explanations: SHAP without the `shap` package
8. Did it learn the hidden rule?
9. Exercise
10. Pitfalls and next steps

Code used: `heavden/ml/evaluate.py` (bootstrap, slices, warning time), `heavden/ml/explain.py` (tests in `tests/test_evaluate_explain.py`).

## 1. Setup: load `@champion` from MLflow

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from heavden.ml import evaluate, explain, tracking, train


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
path = ROOT / "data" / "gold_local" / "patient_hours_baseline.parquet"
if not (ROOT / "data" / "mlflow" / "mlflow.db").exists():
    raise FileNotFoundError("Run notebook 06 first: it registers the @champion model.")
table = pd.read_parquet(path)
tr, va, te = train.time_split(table)

tracking.use_local_store(ROOT / "data" / "mlflow")
champion = mlflow.sklearn.load_model(f"models:/{tracking.LOCAL_MODEL_NAME}@champion")
print("champion base model:", type(champion.base).__name__)

Everything in this notebook evaluates the **registered** model, the exact object that would be served, not a fresh retrain.

## 2. Recap: the single numbers from notebook 06

Thresholds are fixed on **validation** from the alert budget (about 2 alerts per nurse per 12-hour shift, notebook 06) and applied to **test**. Remember the two levels: **per hour** (AUROC, AUPRC: ranking) and **per alert** (what a nurse experiences).

In [ ]:
budget = evaluate.AlertBudget()
y_te = te["label"].to_numpy(int)
scores_va = {"champion": champion.predict_proba(va)[:, 1], "NEWS2": train.news2_scores(va)}
scores_te = {"champion": champion.predict_proba(te)[:, 1], "NEWS2": train.news2_scores(te)}
thresholds = {k: evaluate.threshold_for_alert_budget(va, v, budget) for k, v in scores_va.items()}

point = pd.DataFrame(
    {k: evaluate.evaluate(te, s, thresholds[k], probabilities=False) for k, s in scores_te.items()}
).T
n_enc = te["encounter_id"].nunique()
print(f"test: {len(te):,} patient-hours, {y_te.sum()} positive, {n_enc} encounters")
headline = ["auroc", "auprc", "alerts_per_nurse_shift", "alert_precision", "escalations"]
point[[*headline, "escalations_flagged", "median_warning_hours"]].round(3)

These are **single numbers from one test block**. With fewer than 30 escalations, a different two days could look quite different. How different?

## 3. How sure are we? The cluster bootstrap

The **bootstrap** answers "what if we'd seen a slightly different sample?": draw a new test set *with replacement* from the one we have, recompute the metrics, repeat 1,000 times, and look at the spread.

**The catch:** the hours of one patient are not independent. Neighbouring hours look almost the same, and one escalation produces ~6 positive hours. Resampling individual **rows** pretends we have far more independent evidence than we do. So we resample whole **encounters** (a *cluster* bootstrap). Alerts and escalations belong to one stay, so they resample naturally with it.

Both models are scored on the **same** resample each time, so we can also read off the **difference** between them.

In [ ]:
reps = evaluate.cluster_bootstrap(te, scores_te, thresholds, n_boot=1000, seed=0)
# For comparison only: pretend every row is its own stay (ranking metrics only, since alerts
# need real stays)
as_rows = te.assign(encounter_id=np.arange(len(te)).astype(str))
row_reps = evaluate.cluster_bootstrap(as_rows, scores_te, thresholds, n_boot=1000, seed=0)


def interval_table(r, metrics):
    ci = evaluate.confidence_intervals(r)
    return pd.DataFrame(
        {
            m: [f"{ci.loc[n, ('low', m)]:.2f}-{ci.loc[n, ('high', m)]:.2f}" for n in ci.index]
            for m in metrics
        },
        index=ci.index,
    )


print("95% intervals, resampling ENCOUNTERS (honest):")
display(interval_table(reps, ["auroc", "auprc", "alert_precision", "escalations_flagged"]))
print("95% intervals, resampling ROWS (too narrow):")
display(interval_table(row_reps, ["auroc", "auprc"]))

The row bootstrap's intervals are visibly narrower: that's false confidence. From here on we use the encounter bootstrap.

Now the question that matters: **is the champion better than NEWS2?**

In [ ]:
diff = evaluate.paired_difference(reps, "champion", "NEWS2")
diff.loc[
    ["auroc", "auprc", "alerts_per_nurse_shift", "alert_precision", "escalations_flagged"]
].round(3)

In [ ]:
wide = reps.pivot_table(index=["rep", "metric"], columns="model", values="value")
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, metric in zip(axes, ["auprc", "escalations_flagged"], strict=True):
    d = (wide["champion"] - wide["NEWS2"]).xs(metric, level="metric")
    ax.hist(d, bins=40, color="tab:blue", alpha=0.8)
    ax.axvline(0, color="k", lw=1)
    ax.set_title(f"champion - NEWS2: {metric}")
    ax.set_ylabel("bootstrap resamples")
plt.tight_layout()
plt.show()

**Reading this:**
- **AUPRC** (per hour): the interval for the difference sits well above 0, and the champion wins in every resample. It ranks the hours before an escalation much better.
- **Escalations flagged** (per escalation): the interval sits above 0 too, so the champion flags more of the escalations in advance. But the lower end is close to 0: with 28 escalations, each one is worth about 4 points.
- **Alert precision** (per alert): no real difference. Both scores see an escalation follow roughly 1 in 30 alerts. The model's advantage is in **catching more**, not in fewer false alarms.
- **Alert load**: the champion uses a little more of the budget than NEWS2. NEWS2 scores are whole numbers, so its threshold jumps in steps and can't land exactly on the budget. Part of the extra escalations flagged comes from using the budget more fully, which is a fair advantage of a continuous score, but it's worth stating.
- **AUROC**: the interval touches 0. AUROC is dominated by the many easy negatives, so it separates the two scores less clearly.

A good habit for any model report: **every headline number gets an interval**, and comparisons are made **paired**.

## 4. Alert-load drift: a label-free early warning

Both thresholds were set on validation to stay within 2 alerts per nurse per shift. On test:

In [ ]:
point[["alerts_per_nurse_shift", "high_hour_share"]].assign(
    budget=budget.alerts_per_nurse_per_shift
).round(3)

The champion's alert load on test is a little **above** budget: the test days have slightly sicker-looking patient-hours than validation. NEWS2 stays below it.

**Platform lesson:** the **alert load needs no labels**, so it is known immediately, unlike AUPRC or escalations flagged, which wait 6 hours for outcomes. A drifting alert load is an early warning that the data or the patient mix has changed. Phase 3 tracks it on the Model and Drift dashboard (warn above 2.5 alerts per nurse per shift, act above 3).

## 5. Fairness: slices

A model can look fine overall and fail one group. We split the test set by **site**, **age band**, **sex** and **ward type**, keeping **one shared threshold**. A ward uses one rule for everyone, so differing alert loads between groups are part of what we want to see. An escalation counts in the group of the patient's last hour before it.

In [ ]:
p_te = scores_te["champion"]
slices = {
    "site": te["site_id"],
    "age band": evaluate.age_band(te["age"]),
    "sex": te["sex"],
    "ward type": te["unit_type"],
}
cols = ["patient_hours", "auprc", "alerts_per_nurse_shift", "alert_precision", "escalations"]
slice_tables = {
    name: evaluate.slice_metrics(te, p_te, thresholds["champion"], by)[
        [*cols, "escalations_flagged"]
    ]
    for name, by in slices.items()
}
pd.concat(slice_tables, names=["slice", "group"]).astype(float).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
ages = slice_tables["age band"]
axes[0].bar(ages.index, ages["alerts_per_nurse_shift"].astype(float), color="tab:orange")
axes[0].set_title("alerts per nurse per shift, by age")
axes[1].bar(ages.index, ages["escalations_flagged"].astype(float), color="tab:green")
axes[1].set_title("share of escalations flagged in advance, by age")
for ax in axes:
    ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

**What to look for, and how to read it fairly:**
- **Sites and sex** look similar: about 2 alerts per nurse per shift everywhere, and most escalations flagged. No group is obviously under-served.
- **Age** is where the alert load is very uneven. Nurses caring for patients **85+** would get about **4 alerts per shift**, and only ~2% of those alerts are followed by an escalation; for patients **18-49** it's about 0.2 per shift, but over a quarter are followed by one. Partly that's correct, since age raises real risk in the hidden rule. But it means the alert burden falls on wards with older patients, and most of it is false alarms. That deserves a paragraph in the model card, and a clinician's view.
- **Escalations flagged** is lowest for **50-69** (4 of 6). That's two missed patients: suggestive, not proof.
- **Ward type**: step-down units get the highest alert load with the lowest precision. A candidate for a closer look once we have more data.
- **Small slices are noisy.** Several groups have 3-6 escalations in test, so one more flagged patient swings the share by 15-30 points. Never conclude "unfair" (or "fair") from a slice that small without an interval. The exercise below asks you to compute one.

## 6. Warning time

Catching an escalation is good. Catching it **early** is what gives a nurse time to act. For each escalation in test, we check whether the patient was flagged **High** in the 6 hours before it (each score with its own budget threshold from validation), and if so, how early.

In [ ]:
warnings = {name: evaluate.warning_times(te, s, thresholds[name]) for name, s in scores_te.items()}
summary = pd.DataFrame(
    {
        name: {
            "escalations": len(w),
            "flagged": int(w["caught"].sum()),
            "flagged share": round(w["caught"].mean(), 2),
            "median hours of warning": w["lead_hours"].median(),
            "flagged >= 3 h ahead": int((w["lead_hours"] >= 3).sum()),
        }
        for name, w in warnings.items()
    }
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
bins = np.arange(0.5, 7.5, 1)
for name, w in warnings.items():
    ax.hist(w["lead_hours"].dropna(), bins=bins, alpha=0.6, label=name)
ax.set_xlabel("hours of warning before the escalation (first High hour)")
ax.set_ylabel("escalations")
ax.legend()
plt.tight_layout()
plt.show()

**How the warning time is measured:** an escalation shows up as a run of consecutive positive hours (each hour whose 6-hour look-ahead contains the event). The event happens in the hour after the run's last row, so a High flag at that last row gives about 1 hour of warning, and one 5 rows earlier about 6 hours. Hourly predictions limit precision to ±1 hour.

**Reading it:** the champion catches more escalations and tends to catch them earlier. It sees **trends** (breathing creeping up over 3 hours) before they cross NEWS2's fixed bands. With fewer than 30 escalations, treat the exact counts as indicative.

## 7. Explanations: SHAP without the `shap` package

For each prediction, **SHAP** splits the model's score into one contribution per feature: *this* pushed the risk up by so much, *that* pulled it down. The contributions add up exactly to the model's score (in **log-odds**, the scale models compute in before turning it into a probability).

We don't need the `shap` package for our two candidates:
- **Logistic regression** is a weighted sum, so each feature's contribution is `weight × (its value − the average value in training)`.
- **LightGBM** computes exact tree SHAP itself (`predict(..., pred_contrib=True)`).

`explain.explain` does both and checks nothing is lost:

In [ ]:
expl = explain.explain(champion, te, background=tr)
total = expl.base_value + expl.contributions.sum(axis=1)
print(
    "contributions add up to the model's log-odds:",
    np.allclose(total, explain.raw_log_odds(champion, te)),
)
expl.contributions.iloc[:3, :6].round(3)

### Grouping correlated features

Notebook 06 showed that correlated features split credit oddly (e.g. a *negative* weight on the 6-hour breathing average). SHAP has the same issue at the feature level. The fix is to **sum contributions per group** a nurse would recognise (breathing rate, oxygen saturation, blood pressure, ...). The group total is stable even when the credit inside it is shuffled around.

**Global importance** = the average size of each group's contribution across all test hours:

In [ ]:
importance = expl.by_group().abs().mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importance.plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("mean |contribution| (log-odds)")
ax.set_title("What the champion relies on (test set)")
plt.tight_layout()
plt.show()

### One patient caught early, one missed

Now the per-patient view, the one the AI assistant's `explain_patient_risk` tool will return. We take the escalation the champion caught **earliest**, and one it **missed**.

In [ ]:
w = warnings["champion"]
caught_case = w[w["caught"]].sort_values("lead_hours", ascending=False).iloc[0]
missed_case = w[~w["caught"]].iloc[0] if (~w["caught"]).any() else None


def show_case(case, title):
    stay = te[te["encounter_id"] == case["encounter_id"]].sort_values("prediction_ts")
    end = case["last_hour_before_event"]
    stay = stay[
        (stay["prediction_ts"] > end - pd.Timedelta(hours=18)) & (stay["prediction_ts"] <= end)
    ]
    risk = champion.predict_proba(stay)[:, 1]
    hours = (stay["prediction_ts"] - end) / pd.Timedelta(hours=1) - 1  # hours before the event

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    axes[0].plot(hours, risk, "o-", label="champion risk")
    axes[0].axhline(thresholds["champion"], color="red", ls="--", lw=1, label="alert threshold")
    axes[0].set_xlabel("hours before escalation")
    axes[0].set_ylabel("predicted risk")
    axes[0].set_title(title)
    axes[0].legend()

    row = stay.index[-1]  # the last hour before the event
    groups = explain.explain(champion, stay.loc[[row]], background=tr).by_group().loc[row]
    top = groups.reindex(groups.abs().sort_values().index[-8:])
    axes[1].barh(top.index, top.values, color=np.where(top.values > 0, "tab:red", "tab:blue"))
    axes[1].axvline(0, color="k", lw=1)
    axes[1].set_title("why: contributions 1 h before (red = raises risk)")
    plt.tight_layout()
    plt.show()
    e = explain.explain(champion, stay, background=tr)
    return pd.DataFrame(explain.top_factors(e, stay, row, k=3))


show_case(caught_case, f"caught: first alert {caught_case['lead_hours']:.0f} h ahead")

In [ ]:
if missed_case is not None:
    display(show_case(missed_case, "missed: no alert in the 6 h before"))
else:
    print("the champion caught every escalation in this test block")

The table under each case is what `explain.top_factors` returns: the strongest groups, whether they raise or lower risk, and the specific reading that drove each one. This becomes the payload of the assistant's `explain_patient_risk` tool in Phase 4, which turns it into a sentence like *"Risk is high mainly because breathing rate rose to 25/min and SpO2 fell 3 points over 3 hours."*

**Missed cases are worth studying.** Typically the vitals barely moved before the event (a sudden deterioration), or the patient looked "normal for their age" so the background pulled the score down. No model can warn about a change that doesn't show in the data yet.

## 8. Did it learn the hidden rule?

With real patients we can never know the true cause of an escalation. Here we can: the truth generator (`generator/truth.py`) uses a **hidden rule** with abnormal breathing, SpO2, heart rate, blood pressure and temperature, plus age, number of conditions, COPD and heart failure. Sepsis and cardiac deteriorations both drop blood pressure.

Compare that list with the importance chart above:
- the vital-sign groups the hidden rule uses should rank high, and **they do**;
- **patient background** ranks high too, which is correct: age, conditions and heart failure are in the rule;
- oxygen therapy and alertness contribute little. Remember that in our simulation, oxygen and confusion are *consequences* nurses record, not causes in the rule;
- the NEWS2 total adds little on top, because its parts (the individual vitals) are already in the model.

This sanity check is only possible with synthetic data, and it's one of the reasons we built the generator this way.

Finally, the two candidates explained side by side (a model that relied on very different signals would be a red flag):

In [ ]:
results, _ = train.train_candidates(tr, va)
side = pd.DataFrame(
    {r.name: explain.explain(r.model, te, background=tr).by_group().abs().mean() for r in results}
)
(side / side.sum()).sort_values("logistic_regression", ascending=False).round(3)

Shares of total importance. The two models broadly agree on the big groups (background, blood pressure, SpO2, breathing, heart rate), which raises our trust in both. Where they differ (logistic regression leans a little on monitoring gaps and ward type, LightGBM hardly at all), the signals are weak and either reading is plausible.

## 9. Exercise: does the model work for COPD patients?

COPD patients live with low SpO2, and NEWS2 has a special scale (scale 2) for them. A model could easily treat their "normal" low SpO2 as alarming, or ignore a real drop.

1. slice the test set by the `copd` column with `evaluate.slice_metrics`, for **both** the champion and NEWS2 (each with its own validation threshold)
2. how do alert load and escalations flagged compare between COPD and non-COPD patients?
3. stretch: bootstrap only the COPD rows (`evaluate.cluster_bootstrap` on `te[te["copd"]]`) and report the interval for the champion's escalations flagged. Is there enough data to conclude anything?

In [ ]:
# Try it yourself
copd_table = None

In [ ]:
# Answer (run after trying)
copd_group = te["copd"].map({True: "COPD", False: "no COPD"})
copd_table = pd.concat(
    {
        name: evaluate.slice_metrics(te, s, thresholds[name], copd_group)
        for name, s in scores_te.items()
    },
    names=["model", "group"],
)[
    [
        "patient_hours",
        "alerts_per_nurse_shift",
        "alert_precision",
        "escalations",
        "escalations_flagged",
    ]
]
display(copd_table.astype(float).round(3))

copd = te[te["copd"]]
copd_reps = evaluate.cluster_bootstrap(
    copd, {"champion": champion.predict_proba(copd)[:, 1]}, thresholds, n_boot=500
)
ci = evaluate.confidence_intervals(copd_reps)
ci[[("low", "escalations_flagged"), ("high", "escalations_flagged")]].round(2)

Also compare the **alert load**: even with its COPD scale, NEWS2 raises about twice as many alerts per nurse per shift for COPD patients as for everyone else, so chronically-low-SpO2 patients soak up much of its alert budget, a classic real-world complaint about early-warning scores. The model, which learned each patient's usual level, alerts less on COPD patients than NEWS2 does. And the bootstrap here is a lesson in itself: with only 3 COPD escalations, all flagged, every resample says 100%. **An interval can't rescue a sample that small**; the honest statement is "3 of 3 flagged; too few to judge".

## 10. Pitfalls and next steps

**Pitfall: a single number with no interval.** With rare events, two days of test data give wide intervals. Report them.

**Pitfall: row-level bootstraps.** They ignore that a patient's hours are correlated and make intervals look too tight. Resample patients or encounters.

**Pitfall: counting hours instead of alerts.** A score that flickers around its threshold raises many more alerts than one that stays High. Measure the alert load, compare models at the same load, and monitor it.

**Pitfall: per-slice thresholds to "fix" fairness.** Tuning a separate threshold per group can hide a real problem and is hard to justify clinically. Show the gaps first, then decide with clinicians.

**Pitfall: reading single-feature explanations of correlated inputs.** Group them.

**Where this goes in the platform:**
- the bootstrap intervals and slice tables go into the **model card** and the retraining gates (a challenger must not be worse on any large slice);
- the alert load becomes a label-free monitor (Phase 3);
- `explain.top_factors` becomes the `explain_patient_risk` tool (Phase 4).

**Next notebook:** `08_rag_docs_and_chunking`. The document corpus behind the assistant, how to chunk it and how to search it.